In [1]:
!ray stop

Did not find any active Ray processes.


# Initialization

In [24]:
import os
import torch
import glob
from collections import OrderedDict
import re
import shutil
from pathlib import Path
from accelerate.utils import merge_fsdp_weights

## Checkpoint conversion

In [ ]:
import os, re, torch
from collections import defaultdict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
import torch.distributed._tensor



ckpt_dir   = "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_600/actor"
base_model = "/gpfs/models/huggingface.co/Qwen/Qwen2.5-32B"
out_dir    = "/tmp/Qwen-2.5-32B"
os.makedirs(out_dir, exist_ok=True)

# # -------- 1. load every rank checkpoint --------
regex   = re.compile(r"model_world_size_\d+_rank_(\d+)\.pt")
rank_sd = {}                              # rank → state-dict
for f in os.listdir(ckpt_dir):
    m = regex.match(f)
    if m:
        rank = int(m.group(1))
        rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")

world_size = len(rank_sd)
assert world_size > 0, "no rank files found"

# -------- 2. collect per-param slices --------
slices = defaultdict(list)                # param → [local_tensor per rank]
for rank in range(world_size):
    for k, v in rank_sd[rank].items():
        if isinstance(v, torch.distributed._tensor.DTensor):
            # grab the shard stored on this rank
            slices[k].append(v._local_tensor)     # private attr but works fine
        else:                                     # unsharded params / scalars
            slices[k] = [v] * world_size          # replicate so cat() is no-op

# -------- 3. reassemble full tensors --------
full = {}
for k, parts in slices.items():
    # if more than one shard, concatenate along dim 0
    full[k] = torch.cat(parts, dim=0) if len(parts) > 1 else parts[0]

# -------- 4. drop into an HF model and save --------
cfg   = AutoConfig.from_pretrained(base_model)
model = AutoModelForCausalLM.from_config(cfg)
model.load_state_dict(full, strict=False)
model.save_pretrained(out_dir)
AutoTokenizer.from_pretrained(base_model).save_pretrained(out_dir)

print("✅ merged checkpoint written to", out_dir)


/tmp/ipykernel_50838/567394585.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")


✅ merged checkpoint written to /tmp/Qwen-2.5-32B


In [9]:
import os, re, torch
from collections import defaultdict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
import torch.distributed._tensor



ckpt_dir   = "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS/global_step_600/actor"
base_model = "/gpfs/models/huggingface.co/Qwen/Qwen2.5-32B"
out_dir    = "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS/global_step_600/merged"
os.makedirs(out_dir, exist_ok=True)

# # -------- 1. load every rank checkpoint --------
regex   = re.compile(r"model_world_size_\d+_rank_(\d+)\.pt")
rank_sd = {}                              # rank → state-dict
for f in os.listdir(ckpt_dir):
    m = regex.match(f)
    if m:
        rank = int(m.group(1))
        rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")

world_size = len(rank_sd)
assert world_size > 0, "no rank files found"

# -------- 2. collect per-param slices --------
slices = defaultdict(list)                # param → [local_tensor per rank]
for rank in range(world_size):
    for k, v in rank_sd[rank].items():
        if isinstance(v, torch.distributed._tensor.DTensor):
            # grab the shard stored on this rank
            slices[k].append(v._local_tensor)     # private attr but works fine
        else:                                     # unsharded params / scalars
            slices[k] = [v] * world_size          # replicate so cat() is no-op

# -------- 3. reassemble full tensors --------
full = {}
for k, parts in slices.items():
    # if more than one shard, concatenate along dim 0
    full[k] = torch.cat(parts, dim=0) if len(parts) > 1 else parts[0]

# -------- 4. drop into an HF model and save --------
cfg   = AutoConfig.from_pretrained(base_model)
model = AutoModelForCausalLM.from_config(cfg)
model.load_state_dict(full, strict=False)
model.save_pretrained(out_dir)
AutoTokenizer.from_pretrained(base_model).save_pretrained(out_dir)

print("✅ merged checkpoint written to", out_dir)


/tmp/ipykernel_50838/3248474924.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")


✅ merged checkpoint written to /gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS/global_step_600/merged


In [10]:
import os, re, torch
from collections import defaultdict
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
import torch.distributed._tensor



ckpt_dir   = "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B/global_step_150/actor"
base_model = "/gpfs/models/huggingface.co/Qwen/Qwen2.5-32B"
out_dir    = "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B/global_step_150/merged"
os.makedirs(out_dir, exist_ok=True)

# # -------- 1. load every rank checkpoint --------
regex   = re.compile(r"model_world_size_\d+_rank_(\d+)\.pt")
rank_sd = {}                              # rank → state-dict
for f in os.listdir(ckpt_dir):
    m = regex.match(f)
    if m:
        rank = int(m.group(1))
        rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")

world_size = len(rank_sd)
assert world_size > 0, "no rank files found"

# -------- 2. collect per-param slices --------
slices = defaultdict(list)                # param → [local_tensor per rank]
for rank in range(world_size):
    for k, v in rank_sd[rank].items():
        if isinstance(v, torch.distributed._tensor.DTensor):
            # grab the shard stored on this rank
            slices[k].append(v._local_tensor)     # private attr but works fine
        else:                                     # unsharded params / scalars
            slices[k] = [v] * world_size          # replicate so cat() is no-op

# -------- 3. reassemble full tensors --------
full = {}
for k, parts in slices.items():
    # if more than one shard, concatenate along dim 0
    full[k] = torch.cat(parts, dim=0) if len(parts) > 1 else parts[0]

# -------- 4. drop into an HF model and save --------
cfg   = AutoConfig.from_pretrained(base_model)
model = AutoModelForCausalLM.from_config(cfg)
model.load_state_dict(full, strict=False)
model.save_pretrained(out_dir)
AutoTokenizer.from_pretrained(base_model).save_pretrained(out_dir)

print("✅ merged checkpoint written to", out_dir)


/tmp/ipykernel_50838/1012440967.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  rank_sd[rank] = torch.load(os.path.join(ckpt_dir, f), map_location="cpu")


✅ merged checkpoint written to /gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B/global_step_150/merged


In [1]:
import os
import sys
from contextlib import contextmanager
from copy import deepcopy
from dataclasses import dataclass, field
from enum import Enum
from functools import partial
from math import ceil
from pprint import pprint
from typing import Dict, Type
from uuid import uuid4
from collections import defaultdict

import numpy as np
import torch
from torch.utils.data import RandomSampler, SequentialSampler
from torchdata.stateful_dataloader import StatefulDataLoader
from transformers import AutoProcessor, AutoTokenizer

import hydra
import ray
from omegaconf import OmegaConf, open_dict
from tqdm import tqdm
from codetiming import Timer

# Extend import path and set env vars before project-local imports
sys.path.insert(0, "/gpfs/users/zhangyiqi/srl/verl-val")
os.environ["SGLANG_BLOCK_NONZERO_RANK_CHILDREN"] = "0"

# ─── verl project ──────────────────────────────────────────────────────────────
from verl import DataProto
from verl.third_party.sglang.entrypoint import CustomEngine

from verl.trainer.ppo.ray_trainer import RayPPOTrainer
from verl.trainer.main_ppo import (
    get_custom_reward_fn,
    get_running_jobs,
    log_using_devices,
    get_current_job_id,
    get_available_devices,
)
from verl.trainer.ppo import core_algos
from verl.trainer.ppo.metric_utils import (
    compute_data_metrics,
    compute_throughout_metrics,
    compute_timing_metrics,
    reduce_metrics,
    bootstrap_metric,
    calc_maj_val,
)

from verl.single_controller.base import Worker
from verl.single_controller.ray import (
    RayResourcePool,
    RayWorkerGroup,
    RayClassWithInitArgs,
)
from verl.single_controller.ray.base import create_colocated_worker_cls

from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from verl.protocol import (
    collate_fn as batch_collate_fn,
    pad_dataproto_to_divisor,
    unpad_dataproto,
)
from verl.utils.fs import copy_to_local
from verl.utils.seqlen_balancing import (
    get_seqlen_balanced_partitions,
    log_seqlen_unbalance,
)
from verl.utils.checkpoint.checkpoint_manager import find_latest_ckpt_path
from verl.utils.tracking import Tracking, ValidationGenerationsLogger
from verl.utils.model import compute_position_id_with_mask

import pandas as pd

from transformers import AutoTokenizer

from verl import DataProto
from verl.utils.fs import copy_to_local
from verl.workers.fsdp_workers import ActorRolloutRefWorker
from verl.utils.hdfs_io import makedirs
from verl.single_controller.ray import RayClassWithInitArgs, RayResourcePool, RayWorkerGroup


INFO 09-24 01:08:05 __init__.py:194] No platform detected, vLLM is running on UnspecifiedPlatform


2025-09-24 01:08:07,359 - INFO - flashinfer.jit: Prebuilt kernels not found, using JIT backend


In [2]:
os.environ["ENSURE_CUDA_VISIBLE_DEVICES"] = os.environ.get('CUDA_VISIBLE_DEVICES', '')
ray.init(runtime_env={
    'env_vars': {
        'TOKENIZERS_PARALLELISM': 'true',
        'NCCL_DEBUG': 'WARN',
        'VLLM_LOGGING_LEVEL': 'WARN',
        "PYTHONPATH": "/gpfs/users/zhangyiqi/srl/verl-val",
        # 'RAY_EXPERIMENTAL_NOSET_ROCR_VISIBLE_DEVICES': '1',
    }
})


2025-09-24 01:08:16,732	INFO worker.py:1879 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.12
Ray version:,2.46.0
Dashboard:,http://127.0.0.1:8265


(ActorRolloutRefWorker pid=8781) You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


(ActorRolloutRefWorker pid=8517) Model config after override: Qwen2Config {
(ActorRolloutRefWorker pid=8517)   "_name_or_path": "/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_600/merged",
(ActorRolloutRefWorker pid=8517)   "architectures": [
(ActorRolloutRefWorker pid=8517)     "Qwen2ForCausalLM"
(ActorRolloutRefWorker pid=8517)   ],
(ActorRolloutRefWorker pid=8517)   "attention_dropout": 0.0,
(ActorRolloutRefWorker pid=8517)   "eos_token_id": 151643,
(ActorRolloutRefWorker pid=8517)   "hidden_act": "silu",
(ActorRolloutRefWorker pid=8517)   "hidden_size": 5120,
(ActorRolloutRefWorker pid=8517)   "initializer_range": 0.02,
(ActorRolloutRefWorker pid=8517)   "intermediate_size": 27648,
(ActorRolloutRefWorker pid=8517)   "max_position_embeddings": 131072,
(ActorRolloutRefWorker pid=8517)   "max_window_layers": 64,
(ActorRolloutRefWorker pid=8517)   "model_type": "qwen2",
(ActorRolloutRefWorker pid=8517)   "num_attention_heads": 40,
(ActorRolloutRefWorker pid=8

Loading checkpoint shards: 100%|██████████| 29/29 [00:03<00:00,  8.51it/s]
(ActorRolloutRefWorker pid=8785) You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`. [repeated 7x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
Loading checkpoint shards:  97%|█████████▋| 28/29 [01:21<00:02,  2.84s/it]


...

(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before resume memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before resume memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.0048512GB, 

(ActorRolloutRefWorker pid=8517) 2025-09-24 01:20:17,473 - INFO - flashinfer.jit: Finished loading JIT ops: cascade
(ActorRolloutRefWorker pid=8517) 2025-09-24 01:20:17,627 - INFO - flashinfer.jit: Finished loading JIT ops: cascade
(ActorRolloutRefWorker pid=8517) 2025-09-24 01:20:17,781 - INFO - flashinfer.jit: Finished loading JIT ops: cascade


(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before resume memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before resume memory occupation, GPU memory allocated: 37.005118464GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) Before release memory occupation, GPU memory allocated: 37.0048512GB, reserved: 38.172360704GB
(ActorRolloutRefWorker pid=8517) After release memory occupation, GPU memory allocated: 37.0048512GB, 

(ActorRolloutRefWorker pid=8517) 2025-09-24 01:20:17,930 - INFO - flashinfer.jit: Finished loading JIT ops: cascade


(ActorRolloutRefWorker pid=8517) self.sampling_params={'n': 1, 'max_new_tokens': 16384, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'repetition_penalty': 1.0, 'temperature': 1.0, 'top_k': -1, 'top_p': 1.0, 'ignore_eos': False}
(ActorRolloutRefWorker pid=8517) self.sampling_params={'n': 1, 'max_new_tokens': 16384, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'repetition_penalty': 1.0, 'temperature': 1.0, 'top_k': -1, 'top_p': 1.0, 'ignore_eos': False}
(ActorRolloutRefWorker pid=8517) self.sampling_params={'n': 1, 'max_new_tokens': 16384, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'repetition_penalty': 1.0, 'temperature': 1.0, 'top_k': -1, 'top_p': 1.0, 'ignore_eos': False}


In [3]:
config = OmegaConf.load('/gpfs/users/zhangyiqi/srl/verl-val/verl/trainer/config/generation_dapo.yaml')
from pprint import pprint
from omegaconf import OmegaConf
pprint(OmegaConf.to_container(config, resolve=True))  # resolve=True will eval symbol values
OmegaConf.resolve(config)
local_path = copy_to_local(config.model.path)
from verl.utils import hf_tokenizer
tokenizer = hf_tokenizer(local_path)
processor = AutoProcessor.from_pretrained(local_path)

if config.rollout.temperature == 0.:
    assert config.data.n_samples == 1, 'When temperature=0, n_samples must be 1.'

# read dataset. Note that the dataset should directly contain chat template format (e.g., a list of dictionary)
dataset = pd.read_parquet(config.data.path)
chat_lst = dataset[config.data.prompt_key].tolist()

chat_lst = [chat.tolist() for chat in chat_lst]

tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ray_cls_with_init = RayClassWithInitArgs(cls=ray.remote(ActorRolloutRefWorker), config=config, role='rollout')
resource_pool = RayResourcePool(process_on_nodes=[config.trainer.n_gpus_per_node] * config.trainer.nnodes)
wg = RayWorkerGroup(resource_pool=resource_pool, ray_cls_with_init=ray_cls_with_init)
wg.init_model()


{'actor': {'fsdp_config': {'fsdp_size': -1},
           'strategy': 'fsdp',
           'ulysses_sequence_parallel_size': 1},
 'data': {'batch_size': 512,
          'n_samples': 1,
          'output_path': '/opt/tiger/math_Qwen2-7B-Instruct.parquet',
          'path': '/gpfs/users/zhangyiqi/srl/data/gsm8k_eval.parquet',
          'prompt_key': 'prompt'},
 'model': {'external_lib': None,
           'path': '/gpfs/users/zhangyiqi/srl/ckpts/DAPO-PPO/Qwen2.5-32B-GS-PR-nc/global_step_600/merged'},
 'rollout': {'disable_log_stats': True,
             'do_sample': True,
             'dtype': 'bfloat16',
             'enable_chunked_prefill': True,
             'enforce_eager': True,
             'free_cache_engine': True,
             'gpu_memory_utilization': 0.8,
             'group_shuffle': True,
             'ignore_eos': False,
             'load_format': 'dummy_dtensor',
             'log_prob_max_token_len_per_gpu': 10240,
             'log_prob_micro_batch_size': None,
             'l

[None, None, None, None, None, None, None, None]

In [39]:
ray.shutdown()

In [7]:
from verl.workers.reward_manager import DAPORewardManager
import importlib
import verl.workers.reward_manager

importlib.reload(verl.workers.reward_manager)
val_reward_fn = DAPORewardManager(tokenizer=tokenizer,
                                num_examine=0,
                                compute_score=None,
                                reward_fn_key='data_source')
def _default_compute_score(data_source, solution_str, ground_truth, extra_info=None):
    if data_source == 'openai/gsm8k':
        from . import gsm8k
        res = gsm8k.compute_score(solution_str, ground_truth)
    elif data_source in ['lighteval/MATH', 'DigitalLearningGmbH/MATH-lighteval']:
        from . import math
        res = math.compute_score(solution_str, ground_truth)
        # [Optional] Math-Verify Integration
        # For enhanced accuracy, consider utilizing Math-Verify (https://github.com/huggingface/Math-Verify).
        # Note: Math-Verify needs to be manually installed via pip: `pip install math-verify`.
        # To use it, override the `compute_score` function with the following implementation:

        # from . import math_verify
        # res = math_verify.compute_score(solution_str, ground_truth)
    elif data_source == 'dapo-math' or data_source == 'math_dapo' or data_source.startswith("aime") or data_source == 'orz':
        from verl.utils.reward_score import math_dapo
        res = math_dapo.compute_score(solution_str, ground_truth)
    elif data_source in [
            'numina_aops_forum', 'numina_synthetic_math', 'numina_amc_aime', 'numina_synthetic_amc', 'numina_cn_k12',
            'numina_olympiads'
    ]:
        from . import prime_math
        res = prime_math.compute_score(solution_str, ground_truth)
    elif data_source in ['codecontests', 'apps', 'codeforces', 'taco']:
        from . import prime_code
        res = prime_code.compute_score(solution_str, ground_truth, continuous=True)
    elif data_source in ['hiyouga/geometry3k']:
        from . import geo3k
        res = geo3k.compute_score(solution_str, ground_truth)
    elif data_source in ['kk_logic']:
        from . import kk
        res = kk.compute_score(solution_str, ground_truth)
    else:
        raise NotImplementedError(f"Reward function is not implemented for {data_source=}")

    if isinstance(res, dict):
        return res
    elif isinstance(res, (int, float, bool)):
        return float(res)
    else:
        return float(res[0])
val_reward_fn.compute_score = _default_compute_score

# Partial

## GSM8K

In [6]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [7]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1


test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial GSM8K Accuracy: 0.935557240333586


In [8]:
correct = 0
for i in range(len(sample_scores)):
    if sample_scores[i] > 0:
        correct += 1
print(f"Partial GSM8K Accuracy: {correct / len(sample_scores)}")

Partial GSM8K Accuracy: 0.935557240333586


## MATH 500

In [10]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [11]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4


test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial MATH500 Accuracy: 0.79


## Minerva

In [12]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [13]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial Minerva Accuracy: 0.3014705882352941


## Olympiad

In [22]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [23]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6


test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial Olympiad Accuracy: 0.43768545994065283


In [29]:
test_batch.non_tensor_batch['reward_model']

array([{'ground_truth': '70', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2013', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2069', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '24', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$f(x)=-1$,$f(x)=x+1$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{10}{3}$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$f(x)=2 x$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '508', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{7}{27}$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{1}{3}, 1, 9$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '1680', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '20', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '5', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2', 'style': 'rule-lighteval/MATH_v2'},
 

In [27]:
print(tokenizer.decode(test_batch.batch['responses'][0], skip_special_tokens=True))

To find the smallest positive integer \( N \) such that \( 20N \) is a multiple of 14 and \( 14N \) is a multiple of 20, we need to approach this problem by examining the factors involved.

First, break down the conditions:
1. \( 20N \) is a multiple of 14.
2. \( 14N \) is a multiple of 20.

1. \( 20N \) is a multiple of 14:
- The prime factorization of 20 is \( 2^2 \cdot 5 \).
- The prime factorization of 14 is \( 2 \cdot 7 \).
- For \( 20N \) to be a multiple of 14, \( 20N \) must contain at least \( 2 \cdot 7 \) in its prime factorization.
- Since 20 already contains a factor of 2, the only additional factor \( N \) needs to ensure is the 7.

2. \( 14N \) is a multiple of 20:
- The prime factorization of 14 is \( 2 \cdot 7 \).
- The prime factorization of 20 is \( 2^2 \cdot 5 \).
- For \( 14N \) to be a multiple of 20, \( 14N \) must contain at least \( 2^2 \cdot 5 \) in its prime factorization.
- Since 14 already contains one factor of 2, \( N \) needs to provide at least one more 

In [37]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/amc23_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1280


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1280, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([1280, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial AMC23 Accuracy: 0.63046875


*** SIGTERM received at time=1758649355 on cpu 96 ***
PC: @     0x7ff48aee2e2e  (unknown)  epoll_wait
    @     0x7ff48adff520  (unknown)  (unknown)
[2025-09-24 01:42:35,455 E 49215 49215] logging.cc:496: *** SIGTERM received at time=1758649355 on cpu 96 ***
[2025-09-24 01:42:35,455 E 49215 49215] logging.cc:496: PC: @     0x7ff48aee2e2e  (unknown)  epoll_wait
[2025-09-24 01:42:35,455 E 49215 49215] logging.cc:496:     @     0x7ff48adff520  (unknown)  (unknown)


# GS

## GSM8K

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/gsm8k_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1319


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial GSM8K Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1320, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 1


test_output_gen_batch shape: torch.Size([1319, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial GSM8K Accuracy: 0.935557240333586


In [ ]:
correct = 0
for i in range(len(sample_scores)):
    if sample_scores[i] > 0:
        correct += 1
print(f"Partial GSM8K Accuracy: {correct / len(sample_scores)}")

Partial GSM8K Accuracy: 0.935557240333586


## MATH 500

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/math500_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 500


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial MATH500 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([504, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 4


test_output_gen_batch shape: torch.Size([500, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial MATH500 Accuracy: 0.79


## Minerva

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/minerva_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 272


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial Minerva Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([272, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([272, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial Minerva Accuracy: 0.3014705882352941


## Olympiad

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/olympiad_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 674


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial Olympiad Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([680, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 6


test_output_gen_batch shape: torch.Size([674, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial Olympiad Accuracy: 0.43768545994065283


In [ ]:
test_batch.non_tensor_batch['reward_model']

array([{'ground_truth': '70', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2013', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2069', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '24', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$f(x)=-1$,$f(x)=x+1$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{10}{3}$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$f(x)=2 x$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '508', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{7}{27}$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '$\\frac{1}{3}, 1, 9$', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '1680', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '20', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '5', 'style': 'rule-lighteval/MATH_v2'},
       {'ground_truth': '2', 'style': 'rule-lighteval/MATH_v2'},
 

In [ ]:
print(tokenizer.decode(test_batch.batch['responses'][0], skip_special_tokens=True))

To find the smallest positive integer \( N \) such that \( 20N \) is a multiple of 14 and \( 14N \) is a multiple of 20, we need to approach this problem by examining the factors involved.

First, break down the conditions:
1. \( 20N \) is a multiple of 14.
2. \( 14N \) is a multiple of 20.

1. \( 20N \) is a multiple of 14:
- The prime factorization of 20 is \( 2^2 \cdot 5 \).
- The prime factorization of 14 is \( 2 \cdot 7 \).
- For \( 20N \) to be a multiple of 14, \( 20N \) must contain at least \( 2 \cdot 7 \) in its prime factorization.
- Since 20 already contains a factor of 2, the only additional factor \( N \) needs to ensure is the 7.

2. \( 14N \) is a multiple of 20:
- The prime factorization of 14 is \( 2 \cdot 7 \).
- The prime factorization of 20 is \( 2^2 \cdot 5 \).
- For \( 14N \) to be a multiple of 20, \( 14N \) must contain at least \( 2^2 \cdot 5 \) in its prime factorization.
- Since 14 already contains one factor of 2, \( N \) needs to provide at least one more 

In [ ]:
dataset = RLHFDataset(
    parquet_files=['/gpfs/users/zhangyiqi/srl/data/amc23_eval.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )
dataloader = StatefulDataLoader(
    dataset,
    batch_size=len(dataset),
    collate_fn=collate_fn,
    shuffle=True,
    drop_last=True,
)

dataset len: 1280


In [ ]:
sample_inputs = []
sample_outputs = []
sample_scores = []
reward_extra_infos_dict: dict[str, list] = defaultdict(list)
for batch in dataloader:
    test_batch = DataProto.from_single_dict(batch)
    use_legacy_validation = False
    input_ids = test_batch.batch['input_ids']
    # TODO: Can we keep special tokens except for padding tokens?
    input_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in input_ids]
    sample_inputs.extend(input_texts)
    test_gen_batch = test_batch.pop(
        batch_keys=['input_ids', 'attention_mask', 'position_ids'],
        non_tensor_batch_keys=['raw_prompt_ids'],
    )
    test_gen_batch.meta_info = {
        'eos_token_id': tokenizer.eos_token_id,
        'pad_token_id': tokenizer.pad_token_id,
        'recompute_log_prob': False,
        'do_sample': config.rollout.val_kwargs.do_sample,
        'validate': True,
    }

    # pad to be divisible by dp_size
    test_gen_batch_padded, pad_size = pad_dataproto_to_divisor(test_gen_batch, wg.world_size)
    print(f'test_gen_batch_padded shape: {test_gen_batch_padded.batch["input_ids"].shape}')
    print(f'test_gen_batch_padded meta info: {test_gen_batch_padded.meta_info}')
    print(f'pad_size: {pad_size}')
    test_output_gen_batch_padded = wg.generate_sequences(test_gen_batch_padded)

    # unpad
    test_output_gen_batch = unpad_dataproto(test_output_gen_batch_padded, pad_size=pad_size)
    print(f'test_output_gen_batch shape: {test_output_gen_batch.batch["responses"].shape}')
    print(f'test_output_gen_batch meta info: {test_output_gen_batch.meta_info}')
    print('validation generation end')

    # Store generated outputs
    output_ids = test_output_gen_batch.batch['responses']
    output_texts = [tokenizer.decode(ids, skip_special_tokens=True) for ids in output_ids]
    sample_outputs.extend(output_texts)

    test_batch = test_batch.union(test_output_gen_batch)

    result = val_reward_fn(test_batch, return_dict=True)
    reward_tensor = result["reward_tensor"]
    if "reward_extra_info" in result:
        for key, lst in result["reward_extra_info"].items():
            reward_extra_infos_dict[key].extend(lst)
    scores = reward_tensor.sum(-1).cpu().tolist()
    sample_scores.extend(scores)
    correct = 0
    for i in range(len(sample_scores)):
        if sample_scores[i] > 0:
            correct += 1
    print(f"Partial AMC23 Accuracy: {correct / len(sample_scores)}")

test_gen_batch_padded shape: torch.Size([1280, 2048])
test_gen_batch_padded meta info: {'eos_token_id': 151643, 'pad_token_id': 151643, 'recompute_log_prob': False, 'do_sample': True, 'validate': True}
pad_size: 0
test_output_gen_batch shape: torch.Size([1280, 16384])
test_output_gen_batch meta info: {}
validation generation end
Partial AMC23 Accuracy: 0.63046875
